# My C++ Chatbot (CppBuddy): run the app
Before you start: Runtime -> Change runtime type -> **T4 GPU**.
Run the cells **in order**. You need a free **ngrok** auth token (ngrok.com -> Your Authtoken).
A Serper key (serper.dev) is optional, only for the web-search toggle.
Replies take roughly 20-60 seconds because code answers are long and the T4 is a small GPU.

## 1. Install

In [1]:
!pip install -q -U fastapi uvicorn pyngrok transformers peft accelerate bitsandbytes requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incomp

## 2. Log in to Hugging Face
Your adapter repo is public and the base model needs no approval, so **you can skip this cell**. It is only needed if you make the adapter repo private.

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Qwen2.5-Coder + your adapter
`ADAPTER_ID` is already set to your adapter (`mohitsharma9761/my-cpp-chatbot`).
Want to compare with the plain model? Set `USE_ADAPTER = False` to run the base model with the same C++ system prompt.

In [3]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_ID = "mohitsharma9761/my-cpp-chatbot"
USE_ADAPTER = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_ID) if USE_ADAPTER else base
model.eval()
print("Model ready")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  162MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Model ready


## 4. Chat helper + the chat page

In [4]:
import threading

SYSTEM_PROMPT = "You are CppBuddy, a friendly expert C++ tutor and coding assistant (C++11 to C++23). Explain simply first, then show short, correct, compilable code in ```cpp blocks. Prefer modern C++ (smart pointers, STL, RAII). Point out undefined behavior and common mistakes. If a question is not about C++ or programming, politely steer back to C++."
lock = threading.Lock()

def generate_reply(messages, max_new_tokens=600):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with lock, torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.4, top_p=0.9,   # low temperature = more reliable code
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

HTML_PAGE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CppBuddy</title>
<style>
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, sans-serif; background:#0f1117; color:#e8e8ee; display:flex; flex-direction:column; height:100vh; }
  header { padding:14px 18px; background:#171a23; border-bottom:1px solid #262a36; display:flex; justify-content:space-between; align-items:center; }
  header h1 { margin:0; font-size:18px; }
  header label { font-size:13px; color:#aab; cursor:pointer; }
  #chat { flex:1; overflow-y:auto; padding:18px; display:flex; flex-direction:column; gap:10px; }
  .msg { max-width:88%; padding:10px 14px; border-radius:14px; line-height:1.45; white-space:pre-wrap; word-wrap:break-word; }
  .user { align-self:flex-end; background:#5b5bf0; }
  .bot { align-self:flex-start; background:#222736; }
  pre { margin:8px 0 0; padding:10px; background:#0b0d12; border:1px solid #2d3242; border-radius:8px; overflow-x:auto; white-space:pre; font-family: ui-monospace, Consolas, monospace; font-size:13px; line-height:1.4; }
  .bar { display:flex; gap:6px; margin:4px 0 8px; }
  .small { padding:4px 10px; font-size:12px; border-radius:6px; background:#333a52; }
  .out { color:#9fe3a3; background:#07090d; }
  form { display:flex; gap:8px; padding:12px; background:#171a23; border-top:1px solid #262a36; }
  input[type=text] { flex:1; padding:12px; border-radius:10px; border:1px solid #2d3242; background:#0f1117; color:#e8e8ee; font-size:15px; }
  button { padding:12px 18px; border:0; border-radius:10px; background:#5b5bf0; color:white; font-size:15px; cursor:pointer; }
  button:disabled { opacity:.5; cursor:not-allowed; }
</style>
</head>
<body>
<header>
  <h1>CppBuddy &#128187;</h1>
  <label><input type="checkbox" id="useSearch"> Web search</label>
</header>
<div id="chat"><div class="msg bot">Hey! I'm CppBuddy. Ask me any C++ concept, or paste code to debug or a problem to solve.</div></div>
<form id="form">
  <input type="text" id="input" placeholder="Ask about C++ or paste a coding problem..." autocomplete="off">
  <button id="send" type="submit">Send</button>
</form>
<script>
  const chat = document.getElementById("chat");
  const form = document.getElementById("form");
  const input = document.getElementById("input");
  const send = document.getElementById("send");
  const useSearch = document.getElementById("useSearch");
  let history = [];
  let runEnabled = false;

  fetch("/config").then(r => r.json()).then(c => { runEnabled = !!c.run_enabled; }).catch(() => {});

  // Builds a code block with Copy (and Run, if the server allows it). Uses textContent only, so it is XSS-safe.
  function makeCodeBlock(code, lang) {
    const wrap = document.createElement("div");
    const pre = document.createElement("pre");
    const c = document.createElement("code");
    c.textContent = code;
    pre.appendChild(c);

    const bar = document.createElement("div");
    bar.className = "bar";
    const copy = document.createElement("button");
    copy.type = "button"; copy.className = "small"; copy.textContent = "Copy";
    copy.onclick = () => {
      navigator.clipboard.writeText(code).then(() => {
        copy.textContent = "Copied!";
        setTimeout(() => { copy.textContent = "Copy"; }, 1200);
      });
    };
    bar.appendChild(copy);

    const out = document.createElement("pre");
    out.className = "out";
    out.style.display = "none";

    if (runEnabled && ["cpp", "c++", "cc", "cxx"].includes(lang)) {
      const run = document.createElement("button");
      run.type = "button"; run.className = "small"; run.textContent = "Run";
      run.onclick = async () => {
        run.disabled = true;
        out.style.display = "block";
        out.textContent = "Compiling...";
        try {
          const res = await fetch("/run", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ code: code })
          });
          const d = await res.json();
          out.textContent = (d.stage === "compile" ? "[compile error]\n" : "") + (d.output || "(no output)");
        } catch (err) {
          out.textContent = "Error: " + err;
        }
        run.disabled = false;
      };
      bar.appendChild(run);
    }

    wrap.appendChild(pre);
    wrap.appendChild(bar);
    wrap.appendChild(out);
    return wrap;
  }

  // Splits the reply on ``` fences: even parts are text, odd parts are code.
  function renderInto(el, text) {
    el.textContent = "";
    text.split("```").forEach((part, i) => {
      if (i % 2 === 0) {
        const t = part.replace(/^\n+|\n+$/g, "");
        if (t) { const s = document.createElement("span"); s.textContent = t; el.appendChild(s); }
      } else {
        const nl = part.indexOf("\n");
        const lang = (nl >= 0 ? part.slice(0, nl) : "").trim().toLowerCase();
        const code = (nl >= 0 ? part.slice(nl + 1) : part).replace(/\n$/, "");
        el.appendChild(makeCodeBlock(code, lang));
      }
    });
  }

  function addMsg(text, who) {
    const d = document.createElement("div");
    d.className = "msg " + who;
    chat.appendChild(d);
    renderInto(d, text);
    chat.scrollTop = chat.scrollHeight;
    return d;
  }

  form.addEventListener("submit", async (e) => {
    e.preventDefault();
    const text = input.value.trim();
    if (!text) return;
    input.value = "";
    addMsg(text, "user");
    const thinking = addMsg("Thinking...", "bot");
    send.disabled = true;
    try {
      const res = await fetch(useSearch.checked ? "/search" : "/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ message: text, history: history.slice(-6) })
      });
      const data = await res.json();
      const reply = data.response || "Something went wrong on the server.";
      renderInto(thinking, reply);
      history.push({ role: "user", content: text });
      history.push({ role: "assistant", content: reply });
    } catch (err) {
      thinking.textContent = "Oops, something broke: " + err;
    }
    chat.scrollTop = chat.scrollHeight;
    send.disabled = false;
    input.focus();
  });
</script>
</body>
</html>"""
print("ready")

ready


## 5. Start the server
It will ask for your ngrok token (and an optional Serper key; just press Enter to skip that one).
When it finishes, **click the printed link** and chat!

**About `ENABLE_RUN`:** when `True`, each C++ code block gets a **Run** button that compiles it with `g++` and runs it
(5 second limit, empty stdin). Anyone with your public link could then run code on this Colab machine, **including reading your
Hugging Face token**. Leave it `False` unless you are the only person who has the link.

In [5]:
import json, os, subprocess, tempfile, requests, uvicorn
from getpass import getpass
from typing import List, Optional
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from pyngrok import ngrok

ENABLE_RUN = False   # <-- set True only if you trust everyone who has the link

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = "3CiJicmTDVhvEAZWQXj6QQyi5BR_5NQe9tVEcEPgGawvcQkUd"
except Exception:
    NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ")

try:
    SERPER_API_KEY = userdata.get("SERPER_API_KEY")
except Exception:
    SERPER_API_KEY = getpass("Serper key (optional, press Enter to skip): ").strip() or None

def run_cpp(code, stdin_text=""):
    """Compile with g++ and run in a throwaway temp folder, with time limits."""
    with tempfile.TemporaryDirectory() as d:
        src, exe = os.path.join(d, "main.cpp"), os.path.join(d, "main")
        with open(src, "w") as f:
            f.write(code)
        try:
            c = subprocess.run(["g++", "-std=c++20", "-Wall", "-Wextra", "-O1", src, "-o", exe],
                               capture_output=True, text=True, timeout=30)
        except subprocess.TimeoutExpired:
            return {"stage": "compile", "output": "Compilation timed out."}
        if c.returncode != 0:
            return {"stage": "compile", "output": c.stderr[-3000:]}
        try:
            r = subprocess.run([exe], input=stdin_text, capture_output=True, text=True, timeout=5, cwd=d)
            return {"stage": "run", "output": (r.stdout + r.stderr)[-3000:], "exit_code": r.returncode}
        except subprocess.TimeoutExpired:
            return {"stage": "run", "output": "Timed out after 5 seconds (infinite loop or waiting for input?)."}

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    history: Optional[List[dict]] = []

class RunRequest(BaseModel):
    code: str

def clean_history(h):
    return [m for m in (h or []) if m.get("role") in ("user", "assistant") and isinstance(m.get("content"), str)][-6:]

@app.get("/", response_class=HTMLResponse)
def index():
    return HTML_PAGE

@app.get("/config")
def config():
    return {"run_enabled": ENABLE_RUN}

@app.post("/chat")
def chat(req: ChatRequest):
    messages = clean_history(req.history) + [{"role": "user", "content": req.message}]
    return {"response": generate_reply(messages, 600)}

@app.post("/run")
def run(req: RunRequest):
    if not ENABLE_RUN:
        return {"stage": "disabled", "output": "Running code is disabled on this server."}
    return run_cpp(req.code)

@app.post("/search")
def search(req: ChatRequest):
    if not SERPER_API_KEY:
        return {"response": "No search key set, so web search is off. Turn the toggle off and just chat with me."}
    try:
        r = requests.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
            data=json.dumps({"q": "C++ " + req.message}), timeout=15,
        )
        results = r.json().get("organic", [])[:5]
        context = "\n".join(f"{x.get('title', '')}: {x.get('snippet', '')}" for x in results) or "Nothing found."
    except Exception:
        context = "Web search is unavailable right now."
    prompt = (f"Use these web results as extra context (they may be imperfect):\n---\n{context[:2500]}\n---\n"
              f"Now answer this C++ question: {req.message}")
    return {"response": generate_reply([{"role": "user", "content": prompt}], 700)}

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()

print("\nYOUR CPP CHATBOT IS LIVE:", public_url)
print("Code running is", "ENABLED (careful who you share the link with!)" if ENABLE_RUN else "disabled")
print("(ngrok may show a 'Visit Site' button the first time, just click it)")

Serper key (optional, press Enter to skip): ··········

YOUR CPP CHATBOT IS LIVE: https://footer-cornhusk-awhile.ngrok-free.dev
Code running is disabled
(ngrok may show a 'Visit Site' button the first time, just click it)
